# 03 · Global Ablation Lab

**Purpose.** Ablation-тесты Global: удаление мёртвых/дублирующих фич, без PCA, разное число компонент, выключение модулей, моно-модульные модели.

**What to look for.**
- насколько меняется ранг (Spearman) и уровень (max|diff|) против baseline
- нужен ли PCA и сколько компонент
- как влияет удаление M4 / M5
- стресс-эпизоды под каждой конфигурацией

> Это lab-ноутбук: выводы здесь предварительные, не финальный отчёт. Меняй параметры в ячейке *Parameters* и перезапускай.

In [ ]:
# --- bootstrap: запуск из корня проекта (рядом с data/ и backend/) ---
import sys, os
from pathlib import Path
# найти корень проекта и встать в него
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'processed').is_dir()), _here)
os.chdir(_root)
sys.path.insert(0, str(_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)
import importlib
from lab import utils as u
importlib.reload(u)   # подхватываем правки lab/utils.py без рестарта ядра
print('project root:', _root.name)

## Parameters
Меняй значения здесь и перезапускай ноутбук.

In [ ]:
USE_PCA = True
N_PCA = 10
EMA_ALPHA = 0.05
CONTAMINATION = 0.06
EXTRA_EXCLUDE = []   # добавь сюда фичи, которые хочешь убрать вручную

In [ ]:
d = u.load_final_dataset()
av = u.available_whitelist(d)
base = u.fit_lsi_like_model(d, av, use_pca=USE_PCA, n_pca=N_PCA, ema_alpha=EMA_ALPHA, contamination=CONTAMINATION)
baseline = base['lsi']
print('baseline lsi range', round(baseline.min(),2), round(baseline.max(),2))

### Хелпер: обучить вариант и сравнить с baseline

In [ ]:
def run_variant(name, features=None, **kw):
    feats = features if features is not None else av
    art = u.fit_lsi_like_model(d, feats, use_pca=kw.get('use_pca',USE_PCA),
        n_pca=kw.get('n_pca',N_PCA), ema_alpha=kw.get('ema_alpha',EMA_ALPHA),
        contamination=kw.get('contamination',CONTAMINATION))
    cmp = u.compare_scores(baseline, art['lsi'])
    ep = u.compute_episode_summary(art['date'], art['lsi'])
    row = {'variant': name, 'n_feat': len(art['features']), **{k:round(v,4) if isinstance(v,float) else v for k,v in cmp.items()}}
    for _,r in ep.iterrows():
        row[r['episode']+'_max'] = r.get('max')
    return row, art
print('helper ready')

### Набор ablation-вариантов

In [ ]:
results = []
results.append(run_variant('baseline (26)')[0])
results.append(run_variant('drop flag_end_of_period', [f for f in av if f!='m1_flag_end_of_period'])[0])
results.append(run_variant('drop signal_final', [f for f in av if f!='m1_signal_final'])[0])
results.append(run_variant('drop both dead/dup', [f for f in av if f not in u.DEAD_FEATURES+u.DUPLICATE_FEATURES])[0])
results.append(run_variant('no PCA', use_pca=False)[0])
for nc in [3,5,10,15,len(av)]:
    results.append(run_variant(f'PCA={nc}', n_pca=nc)[0])
mods = u.split_features_by_module(av)
results.append(run_variant('drop M4', [f for f in av if not f.startswith('m4')])[0])
results.append(run_variant('drop M5', [f for f in av if not f.startswith('m5')])[0])
pd.DataFrame(results)

### Моно-модульные exploratory модели (M1-only / M5-only / M4-only)

In [ ]:
mono = []
for mkey in ['m1','m4','m5']:
    feats = mods.get(mkey, [])
    if len(feats) < 2:
        continue
    row, art = run_variant(f'{mkey}-only', feats, n_pca=min(N_PCA,len(feats)))
    mono.append(row)
pd.DataFrame(mono)

### Визуальное сравнение: baseline vs no-PCA vs cleaned

In [ ]:
_, art_nopca = run_variant('no PCA', use_pca=False)
_, art_clean = run_variant('cleaned', [f for f in av if f not in u.DEAD_FEATURES+u.DUPLICATE_FEATURES])
fig, ax = u.plot_lsi_series(base['date'],
    {'baseline': baseline, 'no PCA': art_nopca['lsi'], 'cleaned': art_clean['lsi']},
    episodes=u.STRESS_EPISODES, thresholds=u.DEFAULT_THRESHOLDS, title='Global ablations'); plt.show()

## Notes / Open questions

- Ранг обычно устойчив (Spearman высокий), но уровень (max|diff|) может прыгать — фиксируй обе метрики.
- no-PCA сохраняет ранг? Если да — PCA не несёт уникальной информации.
- Удаление dead/dup безопасно по рангу, но требует перекалибровки порогов (см. 04).